# NB02 — Model Training

**GraphSentry: A Unified GNN Framework for Blockchain Illicit Activity Detection**

This notebook trains GraphSentry's primary model: a 3-layer GCN operating on the
43 anonymised financial features plus an engineered degree feature (44 dims total),
using CC-aware GraphSAINT sampling with validation-based early stopping.

**Requires:** Artefacts from `NB01_data_pipeline.ipynb`

**Produces:**
- `model_a.pth` — trained model weights
- `training_results.json` — metrics, optimal threshold, training history

## 1. Configuration

In [19]:
MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 64
LR = 0.005
LR_FACTOR = 0.5
LR_PATIENCE = 5
HIDDEN_DIM = 128
DROPOUT = 0.5

SAINT_STRATEGY = 'node'
SAINT_BUDGET = 500
SAINT_SAMPLES_PER_EPOCH = 20

SEED = 42

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

## 2. Environment + load artefacts

In [20]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas networkx tqdm scikit-learn

from google.colab import drive
import os, json, pickle, time, random, copy

import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
)

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    stats = json.load(f)

INPUT_DIM = stats['feature_dims']['anonymous'] + 1
print(f"Dataset: {stats['total_ccs']} CCs, {stats['total_nodes']} nodes")
print(f"Input dim: {INPUT_DIM} (43 anonymous + 1 degree)")
print(f"Split: train={stats['split']['train']}, val={stats['split']['val']}, test={stats['split']['test']}")

Using Python 3.12.13 environment at: /usr
Checked 7 packages in 161ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Dataset: 5500 CCs, 39819 nodes
Input dim: 44 (43 anonymous + 1 degree)
Split: train=3850, val=825, test=825


## 3. Dataset construction

Build PyG Data objects for validation and test sets. The engineered degree feature
is appended to the 43 anonymous features, matching the architecture from the MVP.
Training data is fed through the GraphSAINT sampler.

In [21]:
def build_pyg_graph(cc_id):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']

    x = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)
    x = torch.cat([x, deg], dim=1)

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


val_data = [build_pyg_graph(cc_id) for cc_id in meta['val_ids']]
test_data = [build_pyg_graph(cc_id) for cc_id in meta['test_ids']]

val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

print(f"Val:  {len(val_data)} graphs, input dim = {val_data[0].x.size(1)}")
print(f"Test: {len(test_data)} graphs, input dim = {test_data[0].x.size(1)}")

Val:  825 graphs, input dim = 44
Test: 825 graphs, input dim = 44


## 4. Model architecture

In [22]:
class GNNClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


model = GNNClassifier(INPUT_DIM).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Model parameters: 39,810


## 5. CC-aware GraphSAINT sampler

Adapts Zeng et al. (2020) for subgraph classification. After sampling nodes from
the background graph, the sampler expands to include all nodes of any touched CC
(the novel adaptation), applies CC-level normalization to correct sampling bias,
and oversamples illicit CCs within each batch to handle class imbalance.

In [23]:
class SubgraphSAINTSampler:
    def __init__(self, eligible_cc_ids):
        self.eligible_ccs = set(eligible_cc_ids)
        self.N = bg['num_nodes']
        self.M = bg['num_edges']

        self.eligible_illicit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 1]
        self.eligible_licit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 0]
        self.cc_sizes = {cc: len(nodes) for cc, nodes in bg['cc_to_nodes'].items()}

    def sample(self, budget, strategy='node'):
        if strategy == 'node':
            sampled = set(random.sample(range(self.N), min(budget, self.N)))
        elif strategy == 'edge':
            ei = bg['edge_index']
            indices = random.sample(range(self.M), min(budget, self.M))
            sampled = set()
            for idx in indices:
                sampled.add(ei[0, idx].item())
                sampled.add(ei[1, idx].item())
        elif strategy == 'rw':
            adj = bg['adj_list']
            roots = random.sample(range(self.N), min(budget, self.N))
            sampled = set(roots)
            for root in roots:
                current = root
                for _ in range(5):
                    neighbors = adj[current]
                    if not neighbors:
                        break
                    current = random.choice(neighbors)
                    sampled.add(current)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        node_to_cc = bg['node_to_cc']
        touched = {node_to_cc[n] for n in sampled if n in node_to_cc and node_to_cc[n] in self.eligible_ccs}

        if not touched:
            touched = set(random.sample(list(self.eligible_ccs), min(32, len(self.eligible_ccs))))

        touched_illicit = [c for c in touched if meta['cc_label_map'][c] == 1]
        touched_licit = [c for c in touched if meta['cc_label_map'][c] == 0]

        if touched_illicit and touched_licit:
            factor = max(1, len(touched_licit) // len(touched_illicit))
            cc_ids = touched_licit + touched_illicit * factor
        elif not touched_illicit:
            n_inject = max(1, len(touched_licit) // 10)
            injected = random.choices(self.eligible_illicit, k=min(n_inject, len(self.eligible_illicit)))
            cc_ids = touched_licit + injected
        else:
            cc_ids = list(touched)

        data_list = [build_pyg_graph(cc_id) for cc_id in cc_ids]
        norm_weights = self._compute_norms(cc_ids, budget, strategy)
        batch = Batch.from_data_list(data_list)
        return batch, norm_weights

    def _compute_norms(self, cc_ids, budget, strategy):
        weights = []
        for cc_id in cc_ids:
            if strategy == 'node':
                p = 1 - (1 - self.cc_sizes[cc_id] / self.N) ** budget
            elif strategy == 'edge':
                d_c = sum(len(bg['adj_list'][n]) for n in bg['cc_to_nodes'][cc_id])
                p = 1 - (1 - d_c / (2 * self.M + 1)) ** budget
            elif strategy == 'rw':
                p = 1 - (1 - self.cc_sizes[cc_id] / self.N) ** (budget * 5)
            else:
                p = 1.0
            weights.append(1.0 / max(p, 1e-6))
        w = torch.tensor(weights, dtype=torch.float)
        return w * len(w) / w.sum()


sampler = SubgraphSAINTSampler(meta['train_ids'])
test_batch, test_norms = sampler.sample(SAINT_BUDGET, SAINT_STRATEGY)
print(f"Sampler test: {test_batch.num_graphs} CCs, {test_batch.x.size(0)} nodes, "
      f"norm range [{test_norms.min():.3f}, {test_norms.max():.3f}]")

Sampler test: 600 CCs, 5396 nodes, norm range [0.127, 1.433]


## 6. Training infrastructure

In [24]:
def weighted_cross_entropy(logits, targets, norm_weights):
    per_sample = F.cross_entropy(logits, targets, reduction='none')
    return (per_sample * norm_weights.to(logits.device)).mean()


def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1

## 7. Training loop

GraphSAINT sampling with CC-level normalization, validated against the held-out
validation set each epoch. Best model checkpoint is retained via early stopping.

In [25]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
)

best_val_auroc = 0
best_model_state = None
epochs_without_improvement = 0
history = []

print(f"Training: {MAX_EPOCHS} max epochs, patience={PATIENCE}, "
      f"strategy={SAINT_STRATEGY}, budget={SAINT_BUDGET}")
print("-" * 75)

t_start = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0

    for _ in range(SAINT_SAMPLES_PER_EPOCH):
        batch, norm_weights = sampler.sample(SAINT_BUDGET, SAINT_STRATEGY)
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = weighted_cross_entropy(out, batch.y, norm_weights)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / SAINT_SAMPLES_PER_EPOCH

    val_metrics = evaluate(model, val_loader)
    val_auroc = val_metrics['auroc']
    scheduler.step(val_auroc)

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        marker = ' *'
    else:
        epochs_without_improvement += 1
        marker = ''

    lr_now = optimizer.param_groups[0]['lr']
    history.append({
        'epoch': epoch, 'loss': avg_loss,
        'val_auroc': val_auroc, 'val_f1': val_metrics['f1'], 'lr': lr_now,
    })

    if epoch <= 5 or epoch % 5 == 0 or marker:
        print(f"  Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
              f"Val AUROC: {val_auroc:.4f} | Val F1: {val_metrics['f1']:.4f} | "
              f"LR: {lr_now:.6f}{marker}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

t_train = time.time() - t_start
print(f"\nTraining completed in {t_train:.1f}s")
print(f"Best validation AUROC: {best_val_auroc:.4f}")

model.load_state_dict(best_model_state)

Training: 60 max epochs, patience=10, strategy=node, budget=500
---------------------------------------------------------------------------
  Epoch   1 | Loss: 0.6246 | Val AUROC: 0.8381 | Val F1: 0.3361 | LR: 0.005000 *
  Epoch   2 | Loss: 0.4889 | Val AUROC: 0.8689 | Val F1: 0.3962 | LR: 0.005000 *
  Epoch   3 | Loss: 0.4765 | Val AUROC: 0.8485 | Val F1: 0.4055 | LR: 0.005000
  Epoch   4 | Loss: 0.4539 | Val AUROC: 0.8690 | Val F1: 0.3949 | LR: 0.005000 *
  Epoch   5 | Loss: 0.3933 | Val AUROC: 0.8460 | Val F1: 0.3689 | LR: 0.005000
  Epoch  10 | Loss: 0.2478 | Val AUROC: 0.8570 | Val F1: 0.4060 | LR: 0.002500
  Epoch  14 | Loss: 0.2274 | Val AUROC: 0.8726 | Val F1: 0.4223 | LR: 0.002500 *
  Epoch  15 | Loss: 0.2026 | Val AUROC: 0.8517 | Val F1: 0.4153 | LR: 0.002500
  Epoch  20 | Loss: 0.1767 | Val AUROC: 0.8429 | Val F1: 0.3855 | LR: 0.001250

Early stopping at epoch 24 (no improvement for 10 epochs)

Training completed in 64.2s
Best validation AUROC: 0.8726


<All keys matched successfully>

## 8. Test evaluation

Load the best checkpoint and evaluate on the held-out test set. Threshold is tuned
on validation predictions, then applied to test.

In [26]:
val_results = evaluate(model, val_loader)
best_threshold, val_f1_tuned = tune_threshold(val_results['y_true'], val_results['y_prob'])
print(f"Optimal threshold (from val): {best_threshold:.2f} (val F1 = {val_f1_tuned:.4f})")

test_results = evaluate(model, test_loader)
y_true = test_results['y_true']
y_prob = test_results['y_prob']
y_pred = (y_prob >= best_threshold).astype(int)

test_f1 = f1_score(y_true, y_pred, zero_division=0)
test_prec = precision_score(y_true, y_pred, zero_division=0)
test_rec = recall_score(y_true, y_pred, zero_division=0)
test_auroc = test_results['auroc']

cm = confusion_matrix(y_true, y_pred)

print(f"\nTest results @ threshold={best_threshold:.2f}:")
print(f"  AUROC:     {test_auroc:.4f}")
print(f"  F1:        {test_f1:.4f}")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"\nConfusion matrix:")
print(f"  TN={cm[0,0]}, FP={cm[0,1]}")
print(f"  FN={cm[1,0]}, TP={cm[1,1]}")

print(f"\nThreshold sweep on test set:")
print(f"{'Thresh':<8} {'Prec':<8} {'Recall':<8} {'F1':<8}")
print("-" * 36)
for t in np.arange(0.1, 1.0, 0.1):
    preds = (y_prob >= t).astype(int)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f = f1_score(y_true, preds, zero_division=0)
    print(f"{t:<8.1f} {p:<8.4f} {r:<8.4f} {f:<8.4f}")

Optimal threshold (from val): 0.75 (val F1 = 0.4946)

Test results @ threshold=0.75:
  AUROC:     0.8931
  F1:        0.4577
  Precision: 0.3651
  Recall:    0.6133

Confusion matrix:
  TN=670, FP=80
  FN=29, TP=46

Threshold sweep on test set:
Thresh   Prec     Recall   F1      
------------------------------------
0.1      0.2194   0.9333   0.3553  
0.2      0.2472   0.8933   0.3873  
0.3      0.2686   0.8667   0.4101  
0.4      0.2850   0.8133   0.4221  
0.5      0.2969   0.7600   0.4270  
0.6      0.3253   0.7200   0.4481  
0.7      0.3551   0.6533   0.4601  
0.8      0.4112   0.5867   0.4835  
0.9      0.4805   0.4933   0.4868  


## 9. Save artefacts

In [27]:
torch.save(model.state_dict(), os.path.join(PROCESSED_PATH, 'model_a.pth'))

results = {
    'model': 'anonymous_gcn',
    'features': 'anonymous_43dim_plus_degree',
    'input_dim': INPUT_DIM,
    'architecture': f'3-layer GCN, hidden={HIDDEN_DIM}, dropout={DROPOUT}',
    'sampling': {
        'strategy': SAINT_STRATEGY,
        'budget': SAINT_BUDGET,
        'samples_per_epoch': SAINT_SAMPLES_PER_EPOCH,
    },
    'training': {
        'epochs_run': len(history),
        'best_epoch': history[np.argmax([h['val_auroc'] for h in history])]['epoch'],
        'training_time_s': round(t_train, 1),
    },
    'threshold': best_threshold,
    'val_metrics': {
        'auroc': round(best_val_auroc, 4),
        'f1_tuned': round(val_f1_tuned, 4),
    },
    'test_metrics': {
        'auroc': round(test_auroc, 4),
        'f1': round(test_f1, 4),
        'precision': round(test_prec, 4),
        'recall': round(test_rec, 4),
        'confusion_matrix': {
            'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
            'fn': int(cm[1,0]), 'tp': int(cm[1,1]),
        },
    },
    'history': history,
}

with open(os.path.join(PROCESSED_PATH, 'training_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved model_a.pth ({os.path.getsize(os.path.join(PROCESSED_PATH, 'model_a.pth')) / 1024:.1f} KB)")
print(f"Saved training_results.json")
print(f"\nReady for NB03.")

Saved model_a.pth (165.7 KB)
Saved training_results.json

Ready for NB03.
